# soft_prompt_i_probe — variant (i), MLLM-input prompt tuning: LR probe and cost
Variant (i): P learnable rows prepended at the **MLLM's input embedding layer**, the setting Petrov analyses. They traverse all 36 frozen MLLM layers before reaching the diffusion transformer, so unlike variant (ii) the `torch.no_grad()` at `train.py:513` must be removed and gradients travel back through 3B parameters that LoRA never touches.

Both cost and learning rate are expected to differ from variant (ii), which measured 12.14 s/step, 27.2 GB peak, lr 1e-3. Neither transfers: the gradient reaching the prompt now passes through 36 more layers, and activations for those layers must be retained.

Purpose: confirm the prompt is in the graph, find the usable learning rate, and measure seconds per step and peak VRAM. Nothing here is selected on.

## 1. Repo, deps

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    # requirements.txt omits torchdiffeq (omnigen2/transport/integrators.py) and peft (train.py)
    subprocess.run([sys.executable,"-m","pip","install","-q","torchdiffeq"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's is_torchao_available() raises rather than returning False when torchao predates 0.16;
# Colab ships 0.10. torchao is unused by this bf16 path. Guarded, so a no-op once gone.
_ao = subprocess.run([sys.executable,"-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=True)
    print("torchao removed")
sys.path.insert(0, REPO); os.chdir(REPO)

installed.
torchao removed


## 2. Drive, images, splits

In [ ]:
from google.colab import drive, userdata
import json
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                   shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])
subprocess.run(["cp", f"{DRIVE}/screen_exclude_from_training.txt", "/content/"])
print("splits:", [s for s in ["train","val"] if os.path.exists(f"/content/{s}.jsonl")])

Mounted at /content/drive
restored 37063 / 37063
splits: ['train', 'val']


## 3. Prompt length and initialisation
`P` is how many learnable rows the prompt has. Each row is one vector of width 2048, the MLLM's hidden size, so trainable parameters are `P x 2048`.

The rows are initialised from the embeddings of real domain words rather than from noise, so training starts from representations already present in the model's embedding space. Vocabulary initialisation is the standard choice in the prompt-tuning literature (Lester et al. 2021).

`P = 64` for the rate sweep in section 8: a small size well inside the range used in the prompt-tuning literature, and the cheapest point at which to validate the loop. Section 10 re-measures at
soft_prompt_i_probe `P = 668`, the size this variant trains at, across the three candidate rates, and supersedes section 6 for cost, gradient norm and drift. Gradient norms in both sections are the pre-clip totals returned by `clip_grad_norm_`; reading `.grad` after that call censors the column at `max_grad_norm`.

In [ ]:
import torch, re
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

P = 64
MODEL = "OmniGen2/OmniGen2"
SEED = 2233
torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True)

# read only the embedding matrix out of the mllm shards, not the whole 3B model
idx = json.load(open(hf_hub_download(MODEL, "mllm/model.safetensors.index.json")))
key = next(k for k in idx["weight_map"] if k.endswith("embed_tokens.weight"))
emb = load_file(hf_hub_download(MODEL, f"mllm/{idx['weight_map'][key]}"))[key]
print(f"embedding matrix {tuple(emb.shape)} from {idx['weight_map'][key]}")

SEED_WORDS = ("a photo of a laptop tablet smartphone headphones earbuds keyboard computer mouse "
              "game controller gaming console desktop computer setup vr headset on a wooden desk "
              "in an office studio lighting product shot detailed realistic")
ids = tok(SEED_WORDS, add_special_tokens=False).input_ids
reps = (P + len(ids) - 1) // len(ids)
init = emb[torch.tensor((ids * reps)[:P])].clone().float()
assert init.shape == (P, emb.shape[1]), init.shape

os.makedirs("/content/sp", exist_ok=True)
torch.save(init, "/content/sp/init.pt")
json.dump({"P": P, "dim": int(emb.shape[1]), "init_path": "/content/sp/init.pt", "seed": SEED},
          open("/content/soft_prompt.json", "w"))
print(f"P={P}  dim={emb.shape[1]}  trainable params={init.numel():,}  "
      f"= {100*init.numel()/(38*16800*4):.1f}% of LoRA r4")
print(f"seeded from {len(ids)} domain tokens, tiled to {P} rows")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

processor/tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

mllm/model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

embedding matrix (151936, 2048) from model-00001-of-00004.safetensors
P=64  dim=2048  trainable params=131,072  = 5.1% of LoRA r4
seeded from 36 domain tokens, tiled to 64 rows


## 4. Timing data
Training rows in OmniGen2's format, screen targets excluded. 400 rows is enough that the loader never repeats within a probe run.

In [ ]:
rows = [json.loads(l) for l in open("/content/train.jsonl")]
excl = {l.strip() for l in open("/content/screen_exclude_from_training.txt") if l.strip()}
rows = [r for r in rows if r["output_image"] not in excl][:400]
print(f"rows {len(rows)}  (screen targets excluded)")

os.makedirs("/content/data", exist_ok=True)
with open("/content/data/probe.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")

os.makedirs(f"{REPO}/data_configs/train/probe", exist_ok=True)
with open(f"{REPO}/data_configs/train/probe/probe.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/probe.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print("data config written")

rows 400  (screen targets excluded)
data config written


## 5. Patch train.py — prompt at the MLLM input
**A** is unchanged from variant (ii): freeze everything, hand the optimizer the prompt alone.

**B** differs fundamentally. Variant (ii) appended to the encoder's *output* and could leave `torch.no_grad()` in place. Here the prompt sits at the encoder's *input*, so the whole block is replaced: token ids are embedded with the MLLM's own embedding table, the learnable rows are prepended, and the encoder runs with gradients enabled. Its weights stay frozen — only activations are retained, which is where the extra memory goes.

The anchor is the entire six-line `no_grad` block, asserted to match exactly once.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

A = "    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))\n"
A_NEW = "\n".join([
    "    # soft prompting variant (i): the only trainable tensor is a P x hidden prompt prepended at",
    "    # the MLLM input embedding layer. Transformer, VAE and text encoder are frozen.",
    "    import json as _json",
    "    _spc = _json.load(open('/content/soft_prompt.json'))",
    "    model.requires_grad_(False)",
    "    text_encoder.requires_grad_(False)   # no_grad is removed below, so without this the",
    "    vae.requires_grad_(False)            # backward stores .grad for all 3B frozen MLLM params",
    "    SOFT_PROMPT = torch.nn.Parameter(",
    "        torch.load(_spc['init_path'], map_location='cpu').to(accelerator.device, dtype=torch.float32))",
    "    SOFT_PROMPT.requires_grad_(True)",
    "    globals()['SOFT_PROMPT'] = SOFT_PROMPT",
    "    trainable_params = [SOFT_PROMPT]",
    "    logger.info('[soft_prompt] variant(i) P=%d dim=%d trainable=%d' % (",
    "        SOFT_PROMPT.shape[0], SOFT_PROMPT.shape[1], SOFT_PROMPT.numel()))",
]) + "\n"

# variant (i) replaces the whole no_grad block: the prompt sits at the MLLM input, so the
# encoder must run WITH grad enabled for the loss to reach it. Encoder weights stay frozen.
B = "\n".join([
    "                with torch.no_grad():",
    "                    text_feats = text_encoder(",
    "                        input_ids=text_input_ids,",
    "                        attention_mask=text_mask,",
    "                        output_hidden_states=False,",
    "                    ).last_hidden_state",
]) + "\n"
B_NEW = "\n".join([
    "                # variant (i): no_grad removed so gradients reach the prompt through the MLLM",
    "                _emb = text_encoder.get_input_embeddings()(text_input_ids)",
    "                _sp = SOFT_PROMPT.to(_emb.dtype).unsqueeze(0).expand(_emb.shape[0], -1, -1)",
    "                _inp = torch.cat([_sp, _emb], dim=1)",
    "                text_mask = torch.cat([",
    "                    torch.ones(text_mask.shape[0], _sp.shape[1],",
    "                               dtype=text_mask.dtype, device=text_mask.device), text_mask], dim=1)",
    "                text_feats = text_encoder(",
    "                    inputs_embeds=_inp,",
    "                    attention_mask=text_mask,",
    "                    output_hidden_states=False,",
    "                ).last_hidden_state",
    "                if not text_feats.requires_grad:",
    "                    raise RuntimeError('soft prompt is not in the graph - variant (i) patch')",
]) + "\n"

if "variant(i)" in src:
    print("already patched")
else:
    assert src.count(A) == 1, f"anchor A found {src.count(A)} times"
    assert src.count(B) == 1, f"anchor B (the whole no_grad block) found {src.count(B)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A, A_NEW).replace(B, B_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and parses")

patched and parses


## 5a. Loss normalisation
gradient_state.num_steps is 1 on the reference 8-GPU config and 8 here (single GPU, accum 8),
so the shipped normalisation runs at 8x the reference gradient scale. Corrected to match the
LoRA arm, which applies the same replacement. Derivation in lora_train_r{4,8,16} section 5.

In [ ]:
FIX_ACCUM_NORM = True

p = f"{REPO}/train.py"
A4     = ("                loss = (loss * accelerator.gradient_state.num_steps"
          " * accelerator.num_processes) / num_tokens_in_batch\n")
A4_NEW = "                loss = (loss * accelerator.num_processes) / num_tokens_in_batch\n"

src = open(p, encoding="utf-8").read()
if A4_NEW in src:
    print("accum-norm already corrected")
elif not FIX_ACCUM_NORM:
    print("FIX_ACCUM_NORM=False - shipped normalisation retained (8x reference gradient scale)")
else:
    assert src.count(A4) == 1, f"anchor A4 found {src.count(A4)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A4, A4_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("accum-norm corrected and train.py parses")

now = open(p, encoding="utf-8").read()
print(f"  accum-norm fix : {A4_NEW in now}  (FIX_ACCUM_NORM={FIX_ACCUM_NORM})")
print(f"  shipped line   : {A4 in now}")

accum-norm corrected and train.py parses
  accum-norm fix : True  (FIX_ACCUM_NORM=True)
  shipped line   : False


## 6. Gradient and timing probes
Two further anchored edits. The first prints the prompt's gradient norm and its own norm each step: a zero or absent gradient, or a norm that never moves, means the prompt is not being trained — which would produce the arm's predicted result for the wrong reason. The second is the same step timer the LoRA notebooks use.

In [ ]:
src = open(p, encoding="utf-8").read()
G = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
G_NEW = "\n".join([
    # clip_grad_norm_ returns the PRE-clip total norm; reading .grad after the call
    # returns the clipped value, which saturates at max_grad_norm and is not a measurement
    "                    _gnorm = accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)",
    "                    if accelerator.is_main_process:",
    "                        print('[sp] step %d grad_norm %.3e prompt_norm %.4f' % (",
    "                            global_step, float(_gnorm),",
    "                            SOFT_PROMPT.detach().float().norm().item()), flush=True)",
]) + "\n"
if "[sp] step" in src:
    print("gradient probe already patched")
else:
    assert src.count(G) == 1, f"anchor G found {src.count(G)} times"
    open(p, "w", encoding="utf-8").write(src.replace(G, G_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("gradient probe patched")

src = open(p, encoding="utf-8").read()
T = "                global_step += 1\n"
T_NEW = "\n".join([
    "                if accelerator.is_main_process:",
    "                    import time as _t",
    "                    _tg = globals().setdefault('_TIMING', {'prev': None, 'log': []})",
    "                    _now = _t.time()",
    "                    if _tg['prev'] is not None:",
    "                        _tg['log'].append(_now - _tg['prev'])",
    "                        print('[timing] step %d %.3fs peak_vram %.1fGB' % (",
    "                            global_step, _now - _tg['prev'],",
    "                            torch.cuda.max_memory_allocated()/1e9), flush=True)",
    "                    _tg['prev'] = _now",
]) + "\n"
if "[timing]" in src:
    print("timing already patched")
else:
    assert src.count(T) == 1, f"anchor T found {src.count(T)} times"
    open(p, "w", encoding="utf-8").write(src.replace(T, T + T_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("timing patched")

gradient probe patched
timing patched


## 7. Config
`ft_lora.yml` with `lora_ft` left off, the three resolution keys at 512, and `gradient_accumulation_steps: 8` so `global_batch_size: 16` holds on one GPU — the same deviations the LoRA runs record. Learning rate is set per probe run in section 8; everything else is shipped default, including `warmup_t: 500`, so the schedule shape matches the LoRA arm.

In [ ]:
import yaml
cfg = yaml.safe_load(open(f"{REPO}/options/ft_lora.yml"))
cfg["train"]["lora_ft"] = False
cfg["train"]["batch_size"] = 2
cfg["train"]["gradient_accumulation_steps"] = 8
cfg["train"]["global_batch_size"] = 16
cfg["logger"]["checkpointing_steps"] = 100000   # probe writes no checkpoints
cfg["train"]["gradient_checkpointing"] = True   # DiT only; the MLLM is not checkpointed by train.py.
                                                # If this OOMs, that is the first thing to add.
cfg["data"]["max_output_pixels"] = 512*512
cfg["data"]["max_input_pixels"] = [512*512]*4
cfg["data"]["max_side_length"] = 512
cfg["data"]["data_path"] = "data_configs/train/probe/probe.yml"
cfg["logger"]["log_with"] = ["tensorboard"]   # wandb would block on an interactive login
cfg["train"]["max_train_steps"] = 20
cfg["train"]["warmup_t"] = 5        # probe only: ft_lora.yml ships 500, which would keep all
                                    # 20 steps inside warmup at ~2% of the configured rate.
print(f"maximum_text_tokens {cfg['data']['maximum_text_tokens']} | warmup_t {cfg['train'].get('warmup_t')} | "
      f"lora_ft {cfg['train']['lora_ft']}")
os.makedirs("/content/opts", exist_ok=True)

maximum_text_tokens 888 | warmup_t 5 | lora_ft False


## 8. Learning-rate probe
LoRA's `lr: 8e-7` is OmniGen2's tuned value for adapters that multiply into existing weight matrices. A soft prompt is an input: its gradient gets no such amplification, and the prompt-tuning literature trains at rates three to five orders of magnitude higher. Running the arm at 8e-7 would leave the prompt nearly unmoved and produce "soft prompting stays below LoRA" for a reason that has nothing to do with expressiveness.

Six rates, 20 steps each with `warmup_t: 5`, so drift is read over the 15 post-warmup steps. Read the gradient norm (is the prompt in the graph at all), the prompt norm drift (is it actually moving), and the loss trajectory (does it move without diverging). Selection here is on training loss only — no validation, no GPT.

In [ ]:
import time, subprocess
RESULTS = {}
for lr in (1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 1e-1):
    cfg["train"]["learning_rate"] = lr
    cfg["train"]["output_dir"] = f"/content/probe_lr{lr:g}"
    opt = f"/content/opts/probe_lr{lr:g}.yml"
    yaml.safe_dump(cfg, open(opt, "w"), sort_keys=False)
    print(f"\n{'='*70}\nlr = {lr:g}\n{'='*70}", flush=True)
    t0 = time.time()
    r = subprocess.run(["accelerate","launch","--num_processes=1","--mixed_precision=bf16",
                        "train.py","--config",opt], cwd=REPO, capture_output=True, text=True,
                       env={**os.environ, "WANDB_DISABLED": "true"})
    out = r.stdout + r.stderr
    steps  = [float(m) for m in re.findall(r"\[timing\] step \d+ ([\d.]+)s", out)]
    grads  = [m for m in re.findall(r"\[sp\] step \d+ grad_norm (\S+)", out)]
    pnorms = [float(m) for m in re.findall(r"prompt_norm ([\d.]+)", out)]
    # first 12 parsed losses only, a leftover from the 12-step probe config; the sweep now
    # runs 20 steps, so the "first->last" column below is first->12th.
    # rate selection was made on prompt drift with loss as a supporting check only.
    losses = [float(m) for m in re.findall(r"loss[=: ]+([\d.]+)", out)][:12]
    RESULTS[lr] = dict(steps=steps, grads=grads, pnorms=pnorms, losses=losses,
                       rc=r.returncode, wall=time.time()-t0, tail=out[-1500:])
    print(f"rc={r.returncode}  {len(steps)} timed steps  wall {time.time()-t0:.0f}s")
    if r.returncode != 0:
        print(out[-2000:]); break


lr = 0.0001
rc=0  19 timed steps  wall 396s

lr = 0.0003
rc=0  19 timed steps  wall 304s

lr = 0.001
rc=0  19 timed steps  wall 310s

lr = 0.003
rc=0  19 timed steps  wall 302s

lr = 0.01
rc=0  19 timed steps  wall 302s

lr = 0.1
rc=0  19 timed steps  wall 304s


## 9. Result

The `loss first->last` column reads the first and twelfth parsed losses, not the twentieth: the parser's `[:12]` slice predates the move to 20-step sweeps. Rate selection is on prompt drift; loss is a supporting check only.

In [ ]:
import statistics as st
print(f"{'lr':>8s} {'s/step':>8s} {'grad norm':>12s} {'prompt norm':>22s} {'loss first->last':>18s}")
for lr, d in sorted(RESULTS.items()):
    s = f"{st.median(d['steps']):.2f}" if d["steps"] else "-"
    g = d["grads"][-1] if d["grads"] else "-"
    pn = f"{d['pnorms'][0]:.3f} -> {d['pnorms'][-1]:.3f}" if len(d["pnorms"]) > 1 else "-"
    ls = f"{d['losses'][0]:.4f} -> {d['losses'][-1]:.4f}" if len(d["losses"]) > 1 else "-"
    print(f"{lr:>8g} {s:>8s} {g:>12s} {pn:>22s} {ls:>18s}")

print("\nreading it:")
print("  grad norm None or 0      -> the prompt is not in the graph; patch B is wrong")
print("  prompt norm unchanged    -> lr too low; the arm would be undertrained by construction")
print("  loss rising or NaN       -> lr too high")
print("  prompt moving, loss flat or falling -> usable regime")

UPH = 11.77
if RESULTS and any(d["steps"] for d in RESULTS.values()):
    sps = st.median([st.median(d["steps"]) for d in RESULTS.values() if d["steps"]])
    for steps in (2000, 3000):
        h = steps*sps/3600
        print(f"\n{steps} steps: {h:.1f} h = {h*UPH:.0f} units per variant, "
              f"{3*h:.1f} h = {3*h*UPH:.0f} units for three variants")
    print(f"compare LoRA measured 14.98 s/step -> this arm is {sps/14.98:.2f}x LoRA per step")

      lr   s/step    grad norm            prompt norm   loss first->last
  0.0001    12.64    1.800e-02         6.380 -> 6.386   0.0489 -> 0.0354
  0.0003    12.68    9.046e-02         6.380 -> 6.420   0.0489 -> 0.0354
   0.001    12.66    2.697e-02         6.380 -> 6.645   0.0489 -> 0.0354
   0.003    12.69    2.178e+00         6.380 -> 7.696   0.0489 -> 0.0354
    0.01    12.72    5.535e-02        6.380 -> 16.049   0.0489 -> 0.0353
     0.1    12.68    5.604e-03       6.380 -> 147.963   0.0489 -> 0.0360

reading it:
  grad norm None or 0      -> the prompt is not in the graph; patch B is wrong
  prompt norm unchanged    -> lr too low; the arm would be undertrained by construction
  loss rising or NaN       -> lr too high
  prompt moving, loss flat or falling -> usable regime

2000 steps: 7.0 h = 83 units per variant, 21.1 h = 249 units for three variants

3000 steps: 10.6 h = 124 units per variant, 31.7 h = 373 units for three variants
compare LoRA measured 14.98 s/step -> this arm i

## 10. Drift, gradient norm and cost at the trained length
Re-measures at `P = 668`, the size this variant trains at, across the three candidate rates. Twenty steps each. Supersedes section 9 for cost, gradient norm and drift. After the loss-normalisation correction the gradient stays well below `max_grad_norm: 1` at both sizes — 1.0e-01 to 1.8e-01 here — so clipping is inactive and the norm is a measurement rather than a censored value. Rate selection is on prompt drift, with the gradient norm as a supporting diagnostic.

In [ ]:
# Section 10: confirm the rate at the size this variant trains at. The sweep above is P=64;
# drift, gradient norm and cost are re-measured at P=668 across the three candidate rates.
# BEST_LR is not used: the final-step gradient norm varies by 3x or more between runs at the
# same rate, and its argmax lands on rates the drift criterion rejects. Selection is on drift.
RUN_LRS = (3e-4, 1e-3, 3e-3)

init_big = emb[torch.tensor((ids * ((668 // len(ids)) + 1))[:668])].clone().float()
torch.save(init_big, "/content/sp/init_668.pt")
json.dump({"P": 668, "dim": int(emb.shape[1]), "init_path": "/content/sp/init_668.pt", "seed": SEED},
          open("/content/soft_prompt.json", "w"))
print(f"P=668  trainable params={init_big.numel():,} = {100*init_big.numel()/(38*16800*4):.0f}% of LoRA r4\n")

RESULTS_668 = {}
for lr in RUN_LRS:
    cfg["train"]["learning_rate"] = lr
    cfg["train"]["output_dir"] = f"/content/probe_P668_lr{lr:g}"
    opt = f"/content/opts/probe_P668_lr{lr:g}.yml"
    yaml.safe_dump(cfg, open(opt, "w"), sort_keys=False)
    print(f"{'='*70}\nP=668  lr = {lr:g}\n{'='*70}", flush=True)
    r = subprocess.run(["accelerate","launch","--num_processes=1","--mixed_precision=bf16",
                        "train.py","--config",opt], cwd=REPO, capture_output=True, text=True,
                       env={**os.environ, "WANDB_DISABLED": "true"})
    out = r.stdout + r.stderr
    RESULTS_668[lr] = dict(
        steps=[float(m) for m in re.findall(r"\[timing\] step \d+ ([\d.]+)s", out)],
        vram=[float(m) for m in re.findall(r"peak_vram ([\d.]+)GB", out)],
        grads=[float(m) for m in re.findall(r"\[sp\] step \d+ grad_norm (\S+)", out)],
        pnorms=[float(m) for m in re.findall(r"prompt_norm ([\d.]+)", out)],
        rc=r.returncode, tail=out[-1500:])
    print(f"rc={r.returncode}  {len(RESULTS_668[lr]['steps'])} timed steps")
    if r.returncode != 0:
        print(out[-2000:]); break

print(f"\n{'lr':>8s} {'s/step':>8s} {'peak GB':>9s} {'grad norm':>12s} {'prompt norm':>24s} {'drift':>9s}")
for lr, d in sorted(RESULTS_668.items()):
    s  = f"{st.median(d['steps']):.2f}" if d["steps"] else "-"
    v  = f"{max(d['vram']):.1f}" if d["vram"] else "-"
    g  = f"{float(d['grads'][-1]):.3e}" if d["grads"] else "-"
    pn = f"{d['pnorms'][0]:.3f} -> {d['pnorms'][-1]:.3f}" if len(d["pnorms"]) > 1 else "-"
    dr = f"{100*(d['pnorms'][-1]/d['pnorms'][0]-1):+.1f}%" if len(d["pnorms"]) > 1 else "-"
    print(f"{lr:>8g} {s:>8s} {v:>9s} {g:>12s} {pn:>24s} {dr:>9s}")

print("\nreading it:")
_p64 = RESULTS.get(1e-3, {}).get("pnorms")
if _p64 and len(_p64) > 1:
    print(f"  P=64 at lr 1e-3 drifted {100*(_p64[-1]/_p64[0]-1):+.1f}%; "
          f"a comparable figure here confirms the rate transfers")
print("  a grad norm above 1.0 anywhere proves the pre-fix column was censored by max_grad_norm")
print("  drift near 0% -> undertrained by construction; above ~20% -> the prompt is running away")

if any(d["steps"] for d in RESULTS_668.values()):
    sps = st.median([st.median(d["steps"]) for d in RESULTS_668.values() if d["steps"]])
    h = 3000*sps/3600
    print(f"\n3000 steps at P=668: {h:.1f} h = {h*11.77:.0f} units")

# restore the small prompt so a re-run of the sweep above is unchanged
json.dump({"P": P, "dim": int(emb.shape[1]), "init_path": "/content/sp/init.pt", "seed": SEED},
          open("/content/soft_prompt.json", "w"))

P=668  trainable params=1,368,064 = 54% of LoRA r4

P=668  lr = 0.0003
rc=0  19 timed steps
P=668  lr = 0.001
rc=0  19 timed steps
P=668  lr = 0.003
rc=0  19 timed steps

      lr   s/step   peak GB    grad norm              prompt norm     drift
  0.0003    15.80      34.0    1.795e-01         20.610 -> 20.690     +0.4%
   0.001    15.76      34.0    1.803e-01         20.610 -> 21.293     +3.3%
   0.003    15.84      34.0    1.014e-01         20.610 -> 25.534    +23.9%

reading it:
  P=64 at lr 1e-3 drifted +4.1%; a comparable figure here confirms the rate transfers
  a grad norm above 1.0 anywhere proves the pre-fix column was censored by max_grad_norm
  drift near 0% -> undertrained by construction; above ~20% -> the prompt is running away

3000 steps at P=668: 13.2 h = 155 units


## 11. Revert the patch
Leaves the repo clean so the next session re-clones or reuses an unmodified `train.py`.

In [ ]:
subprocess.run(["git","-C",REPO,"checkout","--","train.py"], check=True)
src = open(f"{REPO}/train.py", encoding="utf-8").read()
assert "[soft_prompt]" not in src and "[timing]" not in src and "[sp] step" not in src
assert A4 in src, "loss normalisation not restored"
print("train.py reverted to HEAD")

train.py reverted to HEAD
